# Insulin changes and readmission

This is a companion to the [diabetes readmission prediction project](https://github.com/Dongwoon1d/diabetes-readmission-prediction), using the same dataset and outcome to ask a different question. Run `python run_analysis.py` from the project root before opening this notebook. The UCI data lack insulin-change timestamps and pretreatment glucose severity, so adjusted contrasts are observational associations.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
results = root / 'results'
assert (results / 'effect_estimates.csv').exists(), 'Run python run_analysis.py first'


## Cohort and unadjusted readmission rates

In [ ]:
display(pd.read_csv(results / 'cohort_flow.csv'))
display(pd.read_csv(results / 'group_rates.csv'))

## Four observational contrasts

PSM targets matched Up encounters, while IPTW and AIPW contrast the eligible full cohort under measured adjustment. Their agreement cannot test whether important severity variables were missed.

In [ ]:
effects = pd.read_csv(results / 'effect_estimates.csv')
display(effects)
display(Image(filename=str(root / 'visuals' / 'effect_comparison.png')))

## Diagnostics and assumptions

In [ ]:
diagnostics = json.loads((results / 'diagnostics.json').read_text())
display(pd.Series(diagnostics))
display(pd.read_csv(results / 'balance.csv').sort_values('smd_before', key=abs, ascending=False).head(15))
for name in ('propensity_overlap', 'covariate_balance', 'weight_distribution'):
    display(Image(filename=str(root / 'visuals' / (name + '.png'))))

**Interpretation limit:** Insulin `Up` is a during-encounter summary without a decision timestamp. Some discharge exclusions occur after exposure, and pre-decision clinical severity is unobserved. See `docs/causal_design.md` before describing any result as causal.